# Tarea: el paisaje de pérdida de tu propia red

En esta tarea vas a usar **losslens** para visualizar la función de pérdida alrededor de una red que
**entrenaste en una tarea anterior**, y a explicar lo que ves.

No necesitas saber cómo funciona losslens por dentro. Solo necesitas estas llamadas:

```python
surf = losslens.surface(model, data, loss_fn=nn.CrossEntropyLoss())   # calcular
surf.plot()   surf.plot_3d()   surf.plot_slice()                      # visualizar
surf.summary()                                                        # números
losslens.compare({"A": surf_a, "B": surf_b})                          # comparar
surf.save("a.npz");  surf = losslens.load("a.npz")                    # guardar
```

Documentación completa: https://github.com/iturbideM/losslens

**Qué entregar:** este notebook, ejecutado de principio a fin, con cada *Tu respuesta aquí*
reemplazado por tu respuesta. Las respuestas deben ser breves (unas pocas oraciones) y hacer
referencia a tus gráficas y números.

**Consejos**

- Usa un entorno con GPU (en Colab: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU*).
- Prueba primero con una `resolution` baja (por ejemplo 11) para verificar que todo funciona, y luego
  usa el valor por defecto para tus figuras finales.
- Pásale a losslens datos **sin aumento de datos** (*data augmentation*).
- Guarda cada superficie que calcules con `surf.save(...)`, por si el notebook se reinicia.

## Preparación

In [ ]:
%pip install -q git+https://github.com/iturbideM/losslens

In [ ]:
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import losslens
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# Para probar rápido bajar RES a 11. Para las figuras finales dejar 25 (el valor por defecto)
RES = 25

# Acá se guardan los pesos y las superficies, así si se reinicia el kernel no hay que recalcular todo
os.makedirs("resultados", exist_ok=True)

## Parte 1: Tu modelo y tus datos

Carga (o vuelve a entrenar) la red de tu tarea anterior y los datos con los que fue entrenada.

1. Define la clase de tu modelo y carga sus pesos entrenados.
2. Construye un DataLoader, un Dataset o tensores `(X, y)` con tus datos de **entrenamiento**, sin
   aumento de datos.
3. Escribe la función de pérdida con la que entrenaste.

Si tus batches no son de la forma `(inputs, targets)`, o tu pérdida no es
`loss_fn(model(inputs), targets)`, vas a necesitar `forward=` en lugar de `loss_fn=` (ver la
documentación).

In [ ]:
# Tu modelo
# CNN chica para Fashion MNIST: dos bloques conv + relu + maxpool y después dos capas lineales
class CNN(nn.Module):
    def __init__(self, n_clases=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, n_clases)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 28x28 -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # 14x14 -> 7x7
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


print(CNN())
print("parámetros:", sum(p.numel() for p in CNN().parameters()))

In [ ]:
# Tus datos
# Solo ToTensor y normalización, sin data augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

train_data = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

clases = train_data.classes
print(len(train_data), "imágenes de train,", len(test_data), "de test")
print(clases)

test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

In [ ]:
# Tu función de pérdida
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Entrenamiento del modelo base (si ya existen los pesos los carga)
EPOCHS = 15
LR = 0.02


def entrenar(batch_size, epochs=EPOCHS, lr=LR, seed=0):
    torch.manual_seed(seed)
    model = CNN().to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    loader = DataLoader(train_data, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=(device == "cuda"))
    for ep in range(epochs):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
            n += len(y)
        print(f"  época {ep + 1:2d}/{epochs}  loss train {total / n:.4f}  ({time.time() - t0:.1f}s)")
    return model


@torch.no_grad()
def evaluar(model, loader):
    model.eval()
    total, correctos, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        total += loss_fn(out, y).item() * len(y)
        correctos += (out.argmax(1) == y).sum().item()
        n += len(y)
    return total / n, correctos / n


def entrenar_o_cargar(nombre, batch_size):
    ruta = f"resultados/{nombre}.pt"
    if os.path.exists(ruta):
        model = CNN().to(device)
        model.load_state_dict(torch.load(ruta, map_location=device))
        print("cargado", ruta)
    else:
        print(f"entrenando {nombre} (batch_size={batch_size})")
        model = entrenar(batch_size)
        torch.save(model.state_dict(), ruta)
    model.eval()
    return model


model = entrenar_o_cargar("cnn_bs64", batch_size=64)

# Para medir en train sin shuffle
train_eval_loader = DataLoader(train_data, batch_size=1000, shuffle=False)
loss_tr, acc_tr = evaluar(model, train_eval_loader)
loss_te, acc_te = evaluar(model, test_loader)
print(f"train: loss {loss_tr:.4f}  acc {acc_tr:.4f}")
print(f"test:  loss {loss_te:.4f}  acc {acc_te:.4f}")

## Parte 2: El paisaje de tu modelo

Calcula el paisaje alrededor de tu modelo entrenado con las opciones por defecto, y muestra la
gráfica de contorno, la gráfica 3D y los cortes. Luego imprime el resumen.

In [ ]:
# Calcular la superficie
def superficie(nombre, model, **kwargs):
    """Calcula la superficie o la carga si ya está guardada."""
    ruta = f"resultados/{nombre}.npz"
    if os.path.exists(ruta):
        s = losslens.load(ruta)
        if s.meta["resolution"] == (RES if RES % 2 else RES + 1):
            print("cargada", ruta)
            return s
    s = losslens.surface(model, train_data, loss_fn=loss_fn, resolution=RES, **kwargs)
    s.save(ruta)
    return s


surf = superficie("base_seed0", model)
surf

In [ ]:
# Gráfica de contorno
surf.plot(title="CNN batch 64, scale=1.0")

In [ ]:
# Gráfica 3D
surf.plot_3d(title="CNN batch 64, scale=1.0")

In [ ]:
# Cortes
surf.plot_slice(title="CNN batch 64, cortes por el modelo")

In [ ]:
# Resumen
res_base = surf.summary()

**2.1** Describe la forma del paisaje. ¿Está el modelo entrenado en el fondo de un valle? ¿El
valle es ancho o angosto, redondo o alargado, suave o irregular?

*Tu respuesta aquí.*

**2.2** Usa los números de `summary()` para respaldar tu descripción.

*Tu respuesta aquí.*

## Parte 3: ¿Es una forma real?

Las dos direcciones son aleatorias. Calcula el paisaje de nuevo con otra `seed`, y compara ambos
con `losslens.compare`.

In [ ]:
# Segunda semilla y comparación
surf_s1 = superficie("base_seed1", model, seed=1)
losslens.compare({"seed 0": surf, "seed 1": surf_s1})

**3.1** ¿Se parecen los dos paisajes? ¿Qué te dice eso sobre tu respuesta a la Parte 2?

*Tu respuesta aquí.*

## Parte 4: Acercar y alejar

Calcula el paisaje con al menos otros dos valores de `scale`, uno menor y otro mayor que 1.0, y
grafícalos.

In [ ]:
# Escala menor
surf_chica = superficie("base_scale025", model, scale=0.25)
display(surf_chica.plot(title="scale=0.25"))
display(surf_chica.plot_slice(title="scale=0.25, cortes"))
res_chica = surf_chica.summary()

In [ ]:
# Escala mayor
# Lejos del mínimo la loss crece mucho, así que la miro también en escala log
surf_grande = superficie("base_scale25", model, scale=2.5)
display(surf_grande.plot(log=True, title="scale=2.5 (log10 de la loss)"))
display(surf_grande.plot_slice(log=True, title="scale=2.5, cortes (log10)"))
res_grande = surf_grande.summary()

**4.1** ¿Qué ves en cada escala? ¿Qué valor de `scale` muestra mejor el paisaje de tu modelo, y
por qué?

*Tu respuesta aquí.*

## Parte 5: Comparar dos modelos

Entrena una **segunda versión** de tu modelo que difiera en una sola cosa, y compara los dos
paisajes con `losslens.compare`. Algunas ideas:

- con y sin *weight decay* (u otra regularización, por ejemplo dropout);
- tamaño de batch chico vs. grande;
- con y sin conexiones residuales (*skip connections*) o BatchNorm;
- una versión angosta vs. una ancha de la red;
- un checkpoint temprano vs. el modelo final.

Usa los mismos `scale` y `n_samples` para ambos.

In [ ]:
# Entrenar el segundo modelo
# Lo único que cambia es el tamaño de batch: 2048 en vez de 64 (mismo lr, épocas, optimizador y semilla)
model_bs2048 = entrenar_o_cargar("cnn_bs2048", batch_size=2048)

In [ ]:
# Calcular su paisaje y comparar
# Mismo scale (1.0) y n_samples (2000) que el modelo base
surf_bs2048 = superficie("bs2048_seed0", model_bs2048)
display(surf_bs2048.plot_slice(title="CNN batch 2048, cortes"))
losslens.compare({"batch 64": surf, "batch 2048": surf_bs2048})

**5.1** ¿Qué cambiaste, y qué esperabas ver?

*Tu respuesta aquí.*

**5.2** ¿Qué ves realmente? Usa las gráficas y la tabla del resumen.

*Tu respuesta aquí.*

In [ ]:
# Desempeño en train y test de los dos modelos
print(f"{'modelo':<12}{'loss train':>12}{'acc train':>11}{'loss test':>11}{'acc test':>10}")
for nombre, m in [("batch 64", model), ("batch 2048", model_bs2048)]:
    l_tr, a_tr = evaluar(m, train_eval_loader)
    l_te, a_te = evaluar(m, test_loader)
    print(f"{nombre:<12}{l_tr:>12.4f}{a_tr:>11.4f}{l_te:>11.4f}{a_te:>10.4f}")

**5.3** Compara con el desempeño en test de ambos modelos. En tu caso, ¿un paisaje más plano
va de la mano con un mejor desempeño en test?

*Tu respuesta aquí.*

## Parte 6: Conclusiones

**6.1** En unas pocas oraciones: ¿qué aprendiste sobre tu red a partir de su paisaje de pérdida?

*Tu respuesta aquí.*